<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Feature_engineering_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installing important libraries
!pip install -q -U "scikit-learn>=1.4" "plotly>=5.20" "joblib>=1.4"

In [ ]:
#downloading data from google drive
!gdown 1g4TCUoLtzGd79WYO-bXqx8IkZNvhpeEp

Downloading...
From: https://drive.google.com/uc?id=1g4TCUoLtzGd79WYO-bXqx8IkZNvhpeEp
To: /content/bank-additional-full.csv
100% 5.83M/5.83M [00:00<00:00, 214MB/s]


In [ ]:
#installing important libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score,
    classification_report, confusion_matrix,
)
import joblib
import sklearn

In [ ]:
#import the dataset in the form of csv seperated by ;
bank_data=pd.read_csv("/content/bank-additional-full.csv",sep=";")

In [ ]:
bank_data.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
#doing some preprocessing
y_pre=(bank_data["y"]=="yes").astype(int)
X_pre=bank_data.drop(columns=["y","duration"])
x_tr_pre,x_te_pre,y_tr_pre,y_te_pre=train_test_split(X_pre,y_pre,test_size=0.2,random_state=42)
num_cols_pre=x_tr_pre.select_dtypes(include=np.number).columns.tolist()
cat_cols_pre=x_tr_pre.select_dtypes(exclude=np.number).columns.tolist()
print(f"Bank Marketing (no duration): train={len(x_tr_pre):,}, test={len(x_te_pre):,}")
print(f"Numeric: {len(num_cols_pre)}, Categorical: {len(cat_cols_pre)}")

Bank Marketing (no duration): train=32,950, test=8,238
Numeric: 9, Categorical: 10


In [ ]:
X_pre.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed'],
      dtype='object')

In [ ]:
#preprocessing with duration included
y_actual=(bank_data["y"]=="yes").astype(int)
y_num=len(y_actual)
x_actual=bank_data.drop(columns=["y"])
n_pos,n_neg=int(y_actual.sum()),int((y_actual==0).sum())
print(f"Subscribed people:{n_pos}({(n_pos/y_num)*100:.2f}%)")
print(f"Unsubscribed people:{n_neg}({(n_neg/y_num)*100:.2f}%)")

Subscribed people:4640(11.27%)
Unsubscribed people:36548(88.73%)


In [ ]:
#Seperating columns and rows for the leaky data
num_cols_leaky=x_actual.select_dtypes(include=np.number).columns.tolist()
cat_cols_leaky=x_actual.select_dtypes(exclude=np.number).columns.tolist()
print(f"Numeric: {len(num_cols)}: {num_cols} \n, Categorical: {len(cat_cols)}: {cat_cols}")

Numeric: 10: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'] 
, Categorical: 10: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [ ]:
#importing important libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score,
    classification_report, confusion_matrix,
)
import joblib
import sklearn

#Analyzing f1 score If i include the duration column
X_tr,X_te,Y_tr,Y_te=train_test_split(x_actual,y_actual,test_size=0.2,random_state=42)
leaky_pipe=Pipeline([("preprocess",ColumnTransformer([("num",StandardScaler(),num_cols_leaky),
                    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols_leaky)])),
                    ("clf",LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])

leaky_pipe.fit(X_tr,Y_tr)
Y_pred=leaky_pipe.predict(X_te)
Y_pred_prob=leaky_pipe.predict_proba(X_te)[:,1]
print(classification_report(Y_te,Y_pred))
f1=f1_score(Y_te,Y_pred)


              precision    recall  f1-score   support

           0       0.98      0.86      0.92      7303
           1       0.45      0.89      0.59       935

    accuracy                           0.86      8238
   macro avg       0.71      0.88      0.76      8238
weighted avg       0.92      0.86      0.88      8238



In [ ]:
print(f"The Score of the model is :{f1:.2f}")

The Score of the model is :0.59


In [ ]:
leaky_ap=average_precision_score(Y_te,Y_pred_prob)
print(f"The average precision score is :{leaky_ap:.2f}")


The average precision score is :0.60


In [ ]:
#how the things changes when you drop duration
honest_pipe=Pipeline([("preprocess",ColumnTransformer([("num",StandardScaler(),num_cols_pre),
                    ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols_pre)])),
                    ("clf",LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])

honest_pipe.fit(x_tr_pre,y_tr_pre)
y_pred_honest=honest_pipe.predict(x_te_pre)
y_pred_honest_prob=honest_pipe.predict_proba(x_te_pre)[:,1]
print(classification_report(y_te_pre,y_pred_honest))
f1=f1_score(y_te_pre,y_pred_honest)


              precision    recall  f1-score   support

           0       0.94      0.86      0.90      7303
           1       0.35      0.61      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238



In [ ]:
honest_ap=average_precision_score(y_te_pre,y_pred_honest_prob)
print(f"The average precision score is :{honest_ap:.2f}")

The average precision score is :0.43


In [ ]:
print(f"the f1 score for the same is {f1}")

the f1 score for the same is 0.44357672784068725


In [ ]:
#Sentinel handling
pdays=(X_pre["pdays"]==999).sum()
pdays_remaining=len(X_pre["pdays"])-pdays
print(f"Number of customers not contacted: {pdays}({pdays/(len(X_pre["pdays"]))*100:.2f}%)")
print(f"Number of customers remaining: {pdays_remaining}")

Number of customers not contacted: 39673(96.32%)
Number of customers remaining: 1515


In [ ]:
X_pre = X_pre.copy()
X_pre["was_contacted_before"] = (X_pre["pdays"] != 999).astype(int)
X_pre["pdays_clean"] = X_pre["pdays"].where(X_pre["pdays"] != 999, np.nan)
X_pre["pdays_clean"] = X_pre["pdays_clean"].fillna(X_pre["pdays_clean"].median())

In [ ]:
X_pre.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'was_contacted_before', 'pdays_clean'],
      dtype='object')

In [ ]:
print("After the Fix:")
print(X_pre[["was_contacted_before","pdays_clean"]].describe())

After the Fix:
       was_contacted_before   pdays_clean
count          41188.000000  41188.000000
mean               0.036783      6.000534
std                0.188230      0.733342
min                0.000000      0.000000
25%                0.000000      6.000000
50%                0.000000      6.000000
75%                0.000000      6.000000
max                1.000000     27.000000


In [ ]:
X_pre=X_pre.drop(columns=["pdays"])
X_pre.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'campaign', 'previous', 'poutcome',
       'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m',
       'nr.employed', 'was_contacted_before', 'pdays_clean'],
      dtype='object')

In [ ]:
#Cyclic encoding
month_to_num = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
                "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
m = X_pre["month"].map(month_to_num)
X_pre["month_sin"] = np.sin(2 * np.pi * m / 12)
X_pre["month_cos"] = np.cos(2 * np.pi * m / 12)

In [ ]:
x_tr_pre_fin,x_te_pre_fin,y_tr_pre_fin,y_te_pre_fin=train_test_split(X_pre,y_pre,test_size=0.2,random_state=42)

In [ ]:
#seperaating numerical and categorical columns
num_cols_final=X_pre.select_dtypes(include="number").columns.tolist()
cat_cols_final=X_pre.select_dtypes(include="object").columns.tolist()
print(f"Final numeric ({len(num_cols_final)}):    {num_cols_final}")
print(f"Final categorical ({len(cat_cols_final)}): {cat_cols_final}")

Final numeric (12):    ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'was_contacted_before', 'pdays_clean', 'month_sin', 'month_cos']
Final categorical (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [ ]:
#actual column preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                                 num_cols_final),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cat_cols_final),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocess.set_output(transform="pandas")

pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf",        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])
pipe.fit(x_tr_pre_fin, y_tr_pre_fin)
y_pred = pipe.predict(x_te_pre_fin)
y_proba = pipe.predict_proba(x_te_pre_fin)[:, 1]
print(f"F1 (subscribe class) : {f1_score(y_te_pre_fin, y_pred):.3f}")
print(f"PR-AUC               : {average_precision_score(y_te_pre_fin, y_proba):.3f}")
print(f"ROC-AUC              : {roc_auc_score(y_te_pre_fin, y_proba):.3f}")


F1 (subscribe class) : 0.445
PR-AUC               : 0.425
ROC-AUC              : 0.781
